# Notebook 02: AI Debugging

## Reading and Interpreting Error Logs with AI

This notebook covers how to use AI to debug Python and ML errors effectively. You'll learn to parse stack traces, feed them to LLMs, and build reusable debugging prompt templates.

## 1. The Anatomy of a Python Error

A Python traceback contains:

```
Traceback (most recent call last):      # Stack trace
  File "app.py", line 10, in <module>   # File and line number
    result = divide(10, 0)              # The code that failed
  File "math_utils.py", line 5, in divide  # Where error originated
    return a / b                        # The actual failing operation
ZeroDivisionError: division by zero     # Error type and message
```

### What to Extract for AI

1. **Full traceback** — Don't truncate, AI needs the chain
2. **The failing code** — The function/file mentioned in the traceback
3. **Context** — What you were trying to do
4. **Environment** — Python version, libraries, OS

## 2. Common Python Error Patterns

In [ ]:
# Pattern 1: ImportError / ModuleNotFoundError
# Common causes:
# - Module not installed
# - Wrong import name
# - Circular imports
# - Virtual environment not activated

# BAD: Vague prompt
# "I got an import error"

# GOOD: Specific prompt
# "I'm getting ModuleNotFoundError: No module named 'sklearn'
# I have scikit-learn installed in my venv.
# Python 3.11, Ubuntu 22.04.
# How do I fix this?"

# AI would suggest:
fixes = {
    "sklearn": "pip install scikit-learn  # sklearn is the import name",
    "PIL": "pip install Pillow  # PIL is the import name",
    "cv2": "pip install opencv-python  # cv2 is the import name",
    "yaml": "pip install PyYAML  # yaml is the import name",
    "attr": "pip install attrs  # attr is the import name",
}

for module, fix in fixes.items():
    print(f"{module}: {fix}")

In [ ]:
# Pattern 2: TypeError
# Common causes:
# - Wrong argument types
# - Missing required arguments
# - Incompatible types in operations

# Example TypeError:
def merge_lists(a, b):
    return a + b

# This fails:
# merge_lists([1, 2], "34")  # TypeError: can only concatenate list to list

# AI analysis pattern:
def merge_lists_fixed(a: list, b: list) -> list:
    """Merge two lists, converting non-list inputs."""
    if not isinstance(a, list):
        a = list(a) if hasattr(a, '__iter__') else [a]
    if not isinstance(b, list):
        b = list(b) if hasattr(b, '__iter__') else [b]
    return a + b

print("TypeError prevention via type checking")

In [ ]:
# Pattern 3: AttributeError
# Common causes:
# - Wrong object type
# - Typo in attribute name
# - Object not initialized
# - Missing method in subclass

# Example:
class Dog:
    def __init__(self, name):
        self.name = name
    
    def bark(self):
        return f"{self.name} says Woof!"

# dog = Dog("Rex")
# dog.bark  # AttributeError: 'Dog' object has no attribute 'bark' (missing parentheses)

# AI would catch:
# "Did you mean to call dog.bark()? Missing parentheses."

print("AttributeError: Usually typo or missing parentheses")

In [ ]:
# Pattern 4: KeyError / IndexError
# Common causes:
# - Wrong key name
# - Missing key in dictionary
# - Index out of bounds
# - Data structure assumptions

# Example:
user = {"name": "Alice", "age": 30}
# user["email"]  # KeyError: 'email'

# AI fixes:
# 1. Use .get() with default: user.get("email", "N/A")
# 2. Check key exists: if "email" in user: ...
# 3. Use defaultdict: from collections import defaultdict
# 4. Use try/except KeyError

print("KeyError: Use .get() or check key existence")

In [ ]:
# Pattern 5: ValueError
# Common causes:
# - Invalid argument value
# - Conversion failures
# - Business logic violations

# Example:
# int("abc")  # ValueError: invalid literal for int()
# raise ValueError("Age cannot be negative")

# AI pattern for prevention:
def safe_int(value: str, default: int = 0) -> int:
    """Safely convert to int with fallback."""
    try:
        return int(value)
    except (ValueError, TypeError):
        return default

print(safe_int("42"))      # 42
print(safe_int("abc"))     # 0 (default)
print(safe_int("abc", -1)) # -1

## 3. ML-Specific Error Patterns

### Tensor Shape Mismatches

In [ ]:
# Common ML error: Shape mismatch
# RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x10 and 20x5)

# AI analysis:
ml_errors = {
    "Shape mismatch": {
        "cause": "Input dimensions don't match layer expectations",
        "fix": "Check input shape, first layer input_size, reshape data",
        "debug": "Print tensor shapes at each layer"
    },
    "CUDA out of memory": {
        "cause": "Batch size too large for GPU memory",
        "fix": "Reduce batch size, use gradient accumulation, use mixed precision",
        "debug": "nvidia-smi, torch.cuda.memory_summary()"
    },
    "gradients not computed": {
        "cause": "requires_grad=False or detached tensor",
        "fix": "Ensure inputs require grad, check model.train()",
        "debug": "Check tensor.requires_grad at each step"
    },
    "loss is nan": {
        "cause": "Learning rate too high, data normalization issue",
        "fix": "Lower lr, add gradient clipping, normalize inputs",
        "debug": "Check for inf/nan in data and gradients"
    }
}

for error, info in ml_errors.items():
    print(f"\n{error}:")
    print(f"  Cause: {info['cause']}")
    print(f"  Fix: {info['fix']}")
    print(f"  Debug: {info['debug']}")

### CUDA Errors

In [ ]:
# Common CUDA error patterns and AI suggestions

cuda_errors = [
    {
        "error": "RuntimeError: CUDA error: device-side assert triggered",
        "ai_analysis": "\""Usually means index out of range in embedding layer.
Common causes:
1. vocab_size mismatch in Embedding layer
2. Labels exceed num_classes
3. Data contains unexpected values

Debug steps:
1. Add: torch.cuda.synchronize() after operations
2. Check: data.min(), data.max() against expected range
3. Verify: model's embedding vocab_size matches data\""",
        "fix": "Check embedding layer dimensions, verify label values"
    },
    {
        "error": "RuntimeError: CUDA out of memory",
        "ai_analysis": "\""GPU ran out of VRAM.

Immediate fixes:
1. Reduce batch_size
2. Use torch.cuda.empty_cache()
3. Use mixed precision: torch.cuda.amp

Long-term fixes:
1. Gradient accumulation
2. Model parallelism
3. Gradient checkpointing\""",
        "fix": "Reduce batch size, use gradient accumulation"
    }
]

for item in cuda_errors:
    print(f"Error: {item['error']}")
    print(f"AI Analysis: {item['ai_analysis']}")
    print(f"Fix: {item['fix']}")
    print()

## 4. Building a Debugging Prompt Template

### The PERFECT Debug Prompt

```
CONTEXT:
- Project: [name/description]
- Stack: [language, framework, versions]
- What I was trying to do: [goal]

ERROR:
[paste FULL traceback — do not truncate]

RELEVANT CODE:
[paste the file/function that's failing]

WHAT I'VE TRIED:
- [attempt 1]
- [attempt 2]

REQUEST:
1. What's the root cause?
2. How do I fix it?
3. How do I prevent this class of error?
```

In [ ]:
# Template implementation example

debug_prompt = """
CONTEXT:
- Project: FastAPI user authentication service
- Stack: Python 3.11, FastAPI 0.104, SQLAlchemy 2.0, PostgreSQL
- What I was trying to do: Login endpoint that validates credentials

ERROR:
Traceback (most recent call last):
  File "/app/routes/auth.py", line 45, in login
    user = await db.query(User).filter(User.email == email).first()
  File "/usr/local/lib/python3.11/site-packages/sqlalchemy/ext/asyncio/session.py", line 215, in first
    return await asyncio_fn(*arg, **kw)
sqlalchemy.exc.InvalidRequestError: When using make_transient, the attribute 'password_hash' must not be marked as dirty

RELEVANT CODE:
async def login(email: str, password: str, db: AsyncSession):
    user = await db.query(User).filter(User.email == email).first()
    if not user:
        raise HTTPException(status_code=401)
    if not verify_password(password, user.password_hash):
        raise HTTPException(status_code=401)
    return {"token": create_token(user.id)}

WHAT I'VE TRIED:
- Checked password_hash is not None
- Verified User model has password_hash column

REQUEST:
1. What's the root cause?
2. How do I fix it?
3. How do I prevent this class of error?
"""

print("Example Debug Prompt:")
print(debug_prompt)

## 5. Systematic Debugging Process

### Step-by-Step with AI

1. **Capture** — Copy the full error message
2. **Context** — Identify what code caused it
3. **Query AI** — Use the debugging prompt template
4. **Understand** — AI explains root cause
5. **Fix** — Apply AI's suggestion
6. **Verify** — Run the code again
7. **Prevent** — Ask how to avoid similar errors

In [ ]:
# Example: Systematic debugging

# Step 1: The error
error = """
Traceback (most recent call last):
  File "app.py", line 23, in process_order
    total = sum(item['price'] * item['qty'] for item in items)
  File "app.py", line 23, in <genexpr>
    total = sum(item['price'] * item['qty'] for item in items)
TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'
"""

# Step 2: Context
code = """
def process_order(items):
    total = sum(item['price'] * item['qty'] for item in items)
    return total
"""

# Step 3-7: AI Analysis and Fix
ai_analysis = """
ROOT CAUSE:
One item has price=None. When multiplied by qty (int), it fails.

FIX:
1. Filter None values
2. Add type validation
3. Use default values
"""

fixed_code = """
from typing import Any

def process_order(items: list[dict[str, Any]]) -> float:
    """Calculate order total, skipping invalid items."""
    total = 0.0
    for item in items:
        price = item.get('price')
        qty = item.get('qty', 0)
        
        if price is None or qty is None:
            continue  # Skip invalid items
        
        try:
            total += float(price) * int(qty)
        except (ValueError, TypeError):
            continue  # Skip items with invalid values
    
    return total
"""

print("Error:", error)
print("\nAI Analysis:", ai_analysis)
print("\nFixed Code:", fixed_code)

## 6. Key Takeaways

### Debugging Checklist

- [ ] Copy FULL traceback (don't truncate)
- [ ] Include the failing code
- [ ] State what you were trying to do
- [ ] List what you've already tried
- [ ] Ask for root cause, not just fix
- [ ] Ask how to prevent similar errors

### Common Pitfalls

1. **Truncating tracebacks** — AI needs the full chain
2. **Vague descriptions** — "It doesn't work" isn't helpful
3. **Ignoring environment** — Version mismatches cause many bugs
4. **Not testing the fix** — AI suggestions aren't always right
5. **Skipping prevention** — Fix the root cause, not just the symptom